# Stock Price Prediction

## 1. Problem and objective
Stock prices change every trading day and are difficult to forecast. In this mini project, we use earlier Tesla prices to estimate the next trading day's closing price and compare simple machine-learning models.

## 2. Dataset and download
Use Kaggle's free [Tesla Stock Data (updated till 28 June 2021)](https://www.kaggle.com/datasets/varpit94/tesla-stock-data-updated-till-28jun2021). Sign in to Kaggle, choose **Download**, extract the ZIP, and place the CSV in this project's `data` folder as `Tesla.csv`. For Colab, upload the CSV and change `CSV_PATH` below to its uploaded filename.

Each row is a trading day. `Date` is the trading date; `Open`, `High`, `Low`, and `Close` are the day's prices; `Volume` is the number of shares traded. Some files also have `Adj Close`, which is not needed here. The exact number of rows depends on the CSV edition and is printed after loading.

## 3. Load the data
We use pandas to read the Kaggle CSV. If your file is elsewhere, update `CSV_PATH` to its location.

In [ ]:
# Import tools for tables, plots, models, and evaluation.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# For the project folder, keep the CSV at data/Tesla.csv.
# In Colab, upload the CSV and replace this with its uploaded filename.
CSV_PATH = 'data/Tesla.csv'
df = pd.read_csv(CSV_PATH)

# Standardize headers, then keep the OHLCV columns used in this project.
df.columns = df.columns.astype(str).str.strip().str.title()
required = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
missing_columns = [column for column in required if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing expected columns: {missing_columns}')
df = df[required].copy()
print(f'Dataset size: {df.shape[0]} rows and {df.shape[1]} columns')

## 4. Explore and clean
`head`, `info`, and `describe` show examples, data types, and numeric summaries. We convert dates and prices to the correct types, sort by date, and fill missing numeric values only from earlier days.

In [ ]:
# Inspect a few rows, the column types, and numeric summaries.
display(df.head())
df.info()
display(df.describe())
print('Missing values before cleaning:')
display(df.isna().sum())

# Parse dates and numeric columns; discard invalid dates and sort chronologically.
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for column in required[1:]:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date')
df = df.drop_duplicates(subset='Date', keep='last').reset_index(drop=True)
# Forward-fill uses only a previous observation; drop leading missing values.
df[required[1:]] = df[required[1:]].ffill()
df = df.dropna(subset=required[1:]).reset_index(drop=True)
print('Rows after cleaning:', len(df))

A line plot helps us see how Tesla's closing price changed over time. The market is closed on some days, so the dates are trading dates, not a continuous daily calendar.

In [ ]:
# Plot the historical closing price against the trading date.
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], color='steelblue')
plt.title('Tesla Daily Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing price')
plt.tight_layout()
plt.show()

## 5. Create simple features
Each row uses information available at the end of day *t* to predict the close on day *t + 1*. Moving averages smooth recent prices; daily return measures percentage change; previous close is yesterday's close. The final row has no known next-day target and is removed.

In [ ]:
# Add features using only current or past trading-day data.
df['MA_5'] = df['Close'].rolling(window=5).mean()
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['Daily_Return'] = df['Close'].pct_change()
df['Previous_Close'] = df['Close'].shift(1)

# Shift the outcome back one row so today's features predict tomorrow's close.
df['Target_Close'] = df['Close'].shift(-1)
df['Target_Date'] = df['Date'].shift(-1)
# Class 1 means tomorrow closes higher than today; class 0 means not higher.
df['Target_Up'] = (df['Target_Close'] > df['Close']).astype(int)
feature_columns = required[1:] + ['MA_5', 'MA_20', 'Daily_Return', 'Previous_Close']
df = df.dropna(subset=feature_columns + ['Target_Close', 'Target_Date']).reset_index(drop=True)
print('Model rows after feature creation:', len(df))
display(df[['Date', 'Close', 'MA_5', 'MA_20', 'Daily_Return', 'Previous_Close', 'Target_Close']].head())

## 6. Split by time
We train on the earliest 80% of dates and test on the latest 20%, without shuffling. A random split could let the model learn from future market patterns and then test on the past, which is not a realistic forecast.

In [ ]:
# Separate input features from next-day regression and classification targets.
X = df[feature_columns]
y_price = df['Target_Close']
y_direction = df['Target_Up']

# Keep the later observations entirely for testing.
split_at = int(len(df) * 0.8)
if split_at == 0 or split_at == len(df):
    raise ValueError('Not enough rows for an 80/20 chronological split.')
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y_price.iloc[:split_at], y_price.iloc[split_at:]
direction_train, direction_test = y_direction.iloc[:split_at], y_direction.iloc[split_at:]
test_dates = df['Target_Date'].iloc[split_at:]
print(f'Train rows: {len(X_train)}; test rows: {len(X_test)}')

## 7. Train the models
Linear Regression is a simple baseline. Random Forest averages many decision trees. The optional Decision Tree classifier predicts the direction (up or not up) rather than a price.

In [ ]:
# Fit two regressors to predict the next trading day's closing price.
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
forest_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
forest_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)
forest_predictions = forest_model.predict(X_test)

# Optional classifier: predict whether tomorrow's close is above today's close.
direction_model = DecisionTreeClassifier(max_depth=5, random_state=42)
direction_model.fit(X_train, direction_train)
direction_predictions = direction_model.predict(X_test)

## 8. Evaluate and compare
MAE and RMSE are price errors (lower is better); RMSE penalizes large errors more. R² compares predictions with a simple average-price baseline and can be negative. Accuracy is the fraction of correct up/down predictions.

In [ ]:
# Calculate regression scores on the untouched, later time period.
regression_results = []
for name, predictions in [('Linear Regression', linear_predictions), ('Random Forest', forest_predictions)]:
    regression_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
        'R2': r2_score(y_test, predictions),
    })
regression_comparison = pd.DataFrame(regression_results).sort_values('RMSE')
classifier_accuracy = accuracy_score(direction_test, direction_predictions)
# Add the classifier to the same table; price-error metrics do not apply to it.
classifier_result = pd.DataFrame([{
    'Model': 'Decision Tree Classifier', 'MAE': np.nan, 'RMSE': np.nan,
    'R2': np.nan, 'Accuracy': classifier_accuracy,
}])
regression_comparison['Accuracy'] = np.nan
comparison = pd.concat([regression_comparison, classifier_result], ignore_index=True)
print('Model comparison (regression errors in price units):')
display(comparison.round(3))

# Show classifier accuracy and the counts of actual vs predicted classes.
print('Direction classifier accuracy:', round(classifier_accuracy, 3))
cm = confusion_matrix(direction_test, direction_predictions, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['Not up', 'Up']).plot(cmap='Blues')
plt.title('Decision Tree Direction Confusion Matrix')
plt.tight_layout()
plt.show()

## 9. Visualize predictions and feature importance
Compare both regression models against actual test-period closing prices. The feature-importance chart shows which inputs the Random Forest relied on most; importance is not proof of cause.

In [ ]:
# Plot actual next-day closes and both models' predictions on the test dates.
plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test.to_numpy(), label='Actual', linewidth=2)
plt.plot(test_dates, linear_predictions, label='Linear Regression')
plt.plot(test_dates, forest_predictions, label='Random Forest')
plt.title('Actual vs Predicted Tesla Closing Prices')
plt.xlabel('Next trading day')
plt.ylabel('Closing price')
plt.legend()
plt.tight_layout()
plt.show()

# Display Random Forest's relative importance for each input feature.
importance = pd.Series(forest_model.feature_importances_, index=feature_columns)
importance = importance.sort_values()
importance.plot(kind='barh', figsize=(8, 5), color='teal')
plt.title('Random Forest Feature Importance')
plt.xlabel('Relative importance')
plt.tight_layout()
plt.show()

## 10. Conclusion and limitations
The best regression model in this run is the one with the lowest test RMSE; use the table above rather than assuming one model always wins. Stock markets are noisy and difficult to predict. This small project has no news or sentiment data, and a model can overfit historical patterns. Future work could add more carefully validated predictors or explore an LSTM as an advanced extension.

**Disclaimer:** This project is for learning only and is not financial advice. Predictions are uncertain and should not guide investment decisions.

In [ ]:
# State the best model based on the test-set RMSE measured above.
best_model = comparison.iloc[0]['Model']
best_rmse = comparison.iloc[0]['RMSE']
print(f'Lowest test RMSE: {best_model} (RMSE = {best_rmse:.3f}).')

## Suggested project report structure
1. **Abstract** — short summary of the problem, methods, and findings.
2. **Introduction** — background, problem statement, and objective.
3. **Literature Review** — brief summary of related stock-forecasting approaches.
4. **Methodology** — dataset, feature creation, time split, models, and metrics.
5. **Implementation** — tools, notebook workflow, and key code steps.
6. **Results** — comparison table, plots, and interpretation.
7. **Conclusion** — findings, limitations, and future work.
8. **References** — dataset link and any sources used.

## Likely viva questions
1. **What are you predicting?** The next trading day's closing price, plus an optional up/not-up label.
2. **Why not randomly split the rows?** A random split can mix past and future and make testing unrealistic.
3. **What is a moving average?** The average closing price over a recent fixed number of trading days.
4. **What does MAE tell us?** The average absolute size of prediction errors in price units.
5. **Why compare with Random Forest?** It can learn nonlinear patterns by combining many decision trees.
6. **Can this model guarantee tomorrow's price?** No. Markets are uncertain, and this educational model omits many influences.